In [1]:
# ===== Robust Alpha Vantage loader + CHF risk functions =====

import os, io, time, json, hashlib, pathlib
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
import time
import re
from urllib.parse import urlparse
load_dotenv()

EOD_API = os.getenv("EOD_API")

# --- guard API ticker early ---
if not EOD_API or not isinstance(EOD_API, str):
    raise RuntimeError(
        "one or more api kays not found")

CACHE_DIR = pathlib.Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

today = time.strftime("%Y-%m-%d")
START = '2020-01-01'  # global start date for all fetches

# Module-level debug flag (no new function args). Set RISK_DEBUG=1 in env to enable verbose diagnostics.
DEBUG = os.getenv("RISK_DEBUG", "0") in ("1", "true", "True", "TRUE")

In [2]:
def _cache_path( ticker: str) -> pathlib.Path:
    safe_ticker = ticker.replace("/", "_").replace(":", "_").replace(" ", "_")
    fname = f"{safe_ticker}_{START}.csv"
    return CACHE_DIR / fname

def _is_fresh(path: pathlib.Path, max_age: int) -> bool:
    if not path.exists():
        print(f"Cache file {path} does not exist")
        return False
    age_hours = (time.time() - path.stat().st_mtime) / 3600
    print(f"Cache file {path} age: {age_hours:.2f} hours (max age {max_age} hours)")
    result = age_hours <= max_age
    print(f'Is fresh: {result}')
    return result

def _looks_like_json(payload: bytes) -> bool:
    return payload.lstrip()[:1] in (b"{", b"[")

def fetch_csv_robust(url: str, params: dict, ticker: str, max_age: int = 24) -> pd.DataFrame:
    """
        Robust CSV fetch with:
      - on-disk cache (TTL),
      - JSON throttle/error detection (does NOT overwrite cache),
      - atomic write on success.
        Returns a parsed DataFrame (index on first column).
        """
    print(f"looking at {ticker} ...")
    path = _cache_path( ticker)

    # if cache is fresh return it
    if _is_fresh(path, max_age):
        print(f"Using cached data for {ticker}")
        df = pd.read_csv(path, header=0, parse_dates=[0], index_col=0).sort_index()
        return df
    print(f"Fetching {ticker} ...")
    resp = requests.get(url, params=params)
        
    resp.raise_for_status()
    raw = resp.content


    # Detect JSON throttle/error; do not poison cache
    if _looks_like_json(raw):
        # Try to show a concise message
        try:
            msg = json.loads(raw.decode("utf-8", errors="ignore"))
        except Exception:
            msg = {"body_head": raw[:200].decode("utf-8", errors="ignore")}
        raise RuntimeError(f"... returned JSON (throttle/error) for :{ticker} -> {str(msg)[:180]}")

    # Parse CSV and normalize
    df = pd.read_csv(io.BytesIO(raw), header=0, parse_dates=[0], index_col=0).sort_index()

    print(' columns:', list(df.columns))



    # if fetching USD/xxx set dates back 1 day
    

    print('writing new cache file')
    # Atomic-ish write
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        f.write(raw)
    os.replace(tmp, path)
    return df

def _pick_close_column(df: pd.DataFrame) -> pd.Series:
    """Pick the most appropriate close-like column.
    Prefer adjusted_close when present; else fall back to close. Case-insensitive.
    Returns a float Series.
    """
    if df is None or df.empty:
        raise ValueError("Empty DataFrame passed to _pick_close_column")
    colmap = {c.lower(): c for c in df.columns}
    s = None
    if "adjusted_close" in colmap:
        s = df[colmap["adjusted_close"]]
        # If close also present and differs, keep adjusted_close but note it
        if "close" in colmap:
            try:
                a = pd.to_numeric(s, errors="coerce")
                b = pd.to_numeric(df[colmap["close"]], errors="coerce")
                if not a.fillna(method="ffill").equals(b.fillna(method="ffill")):
                    print("note: adjusted_close != close; using adjusted_close")
            except Exception:
                pass
    elif "adj_close" in colmap:
        s = df[colmap["adj_close"]]
    elif "close" in colmap:
        s = df[colmap["close"]]
    else:
        raise ValueError("DataFrame does not contain 'adjusted_close' or 'close' column.")
    return pd.to_numeric(s, errors="coerce")


def _sort_cols(df):
    """Normalize time index and return a float close-like Series.
    No asset- or currency-specific adjustments here.
    """
    if not df.index.is_monotonic_increasing: 
        print('index wasnt sorted')
        df = df.sort_index()
    df = df[~df.index.duplicated(keep='last')]
    s = _pick_close_column(df).astype('float64')
    return s


def shift_usd_fx_next_day(fx_series: pd.Series) -> pd.Series:
    """
    Given a daily USD/CHF Series (index is dates), shift by -1 so that
    the value at date T comes from T+1. Leaves non-USD series unchanged
    if you choose to guard externally by currency.
    """
    print('shifting fx series()')
    if not isinstance(fx_series, pd.Series):
        raise TypeError("fx_series must be a pandas Series")
    return fx_series.shift(-1)


In [3]:
# ticker = 'USDCHF.FOREX'
# params = {
#     'from': START,  # EODHD uses from/to
#     'to': today,
#     'api_token': EOD_API   
# }
# url = (f'https://eodhd.com/api/eod/{ticker}')
# # 

# df_test = fetch_csv_robust(url, params=params, ticker=ticker,max_age=100)

# print(df_test.head())
# s = _sort_cols(df_test)


# print(s.head())

In [14]:
def build_returns_matrix_in_chf(
    holdings: list[dict],
    lookback_days: int = 252,
    max_age: int = 24,
    no_fx: bool = False,
    DEBUG: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Build CHF daily returns matrix for the provided holdings.

        holdings: list of dicts with tickers:
      - name: row/column label in outputs
      - symbol: EODHD symbol (e.g., 'SWDA.LON', 'IBM')
      - ccy: base currency of the asset price series (GBP/USD/EUR/CHF)
      - gbx: bool; if True, divide close by 100.0 (LSE pence)
      - position: number of shares held (float)

        Returns:
      rets_df: DataFrame of CHF log returns, T x N
      prices_df: DataFrame of CHF closes, T x N
      weights: Series aligned to columns in rets_df
    """

    # Pre-fetch FX once per currency (excluding CHF)
    fx_map: dict[str, pd.Series] = {}
    needed_ccy = sorted({h["ccy"].upper() for h in holdings if h["ccy"].upper() != "CHF"})
    for ccy in needed_ccy:
        ticker = f'{ccy}CHF.FOREX'
        params = {
            'from': START, 
            'to': today,
            'api_token': EOD_API    
        }
        url_fx = f'https://eodhd.com/api/eod/{ticker}'
        
        # Fetch EODHD daily FX and build a Series
        fx_df = fetch_csv_robust(url_fx, params=params, ticker=ticker, max_age=max_age)

        # Normalize and pick close
        fx_s = _sort_cols(fx_df).rename(f"{ccy}CHF")

        if (ccy == "USD" and not no_fx):
            fx_s = shift_usd_fx_next_day(fx_s)
            if DEBUG:
                print("Applied USDCHF T+1 shift")

        fx_map[f'{ccy}CHF'] = fx_s
        if DEBUG:
            try:
                print(f"FX {ccy}CHF: {fx_s.index.min().date()} → {fx_s.index.max().date()} ({len(fx_s)} rows)")
            except Exception:
                print(f"FX {ccy}CHF length: {len(fx_s)} rows")

    if DEBUG:
        print(f"FX keys loaded: {list(fx_map.keys())}")

    # Build CHF close series per asset
    chf_close = {}
    series_ranges: dict[str, tuple[pd.Timestamp, pd.Timestamp, int]] = {}
    if DEBUG:
        print("Per-asset CHF ranges:")
    for h in holdings:
        name  = h["name"]
        sym   = h["symbol"]
        ccy   = h["ccy"].upper()
        gbx   = h["gbx"]

        url_px = f'https://eodhd.com/api/eod/{sym}'

        px_df = fetch_csv_robust(url_px, params=params, ticker=sym, max_age=max_age)

        # Normalize, de-dup, and pick close-like series
        close_local = _sort_cols(px_df)
        if gbx:
            close_local = close_local / 100.0

        if (ccy == "CHF") or no_fx:
            close_chf = close_local.rename(name)
        else:
            fx = fx_map[f'{ccy}CHF']
            # Align FX to price dates and forward-fill FX only (never prices)
            before = fx.reindex(close_local.index)
            fx_aligned = before.ffill()
            filled = int(before.isna().sum() - fx_aligned.isna().sum())
            if DEBUG and filled > 0:
                print(f"    {name}: FX ffill filled {filled} missing FX points on price dates")
            close_chf = (close_local * fx_aligned).dropna().rename(name)

        chf_close[name] = close_chf
        if DEBUG and not close_chf.empty:
            series_ranges[name] = (close_chf.index.min(), close_chf.index.max(), len(close_chf))
            # Stream the per-asset range as we compute it
            s, e, n = series_ranges[name]
            try:
                print(f"  {name}: {s.date()} → {e.date()} ({n} rows)")
            except Exception:
                print(f"  {name}: {n} rows")

    # Align on common dates and restrict to lookback window
    prices_df = pd.DataFrame(chf_close).dropna()
   
    if DEBUG and not prices_df.empty:
        print(
            f"Common window: {prices_df.index.min().date()} → {prices_df.index.max().date()} "
            f"({len(prices_df)} rows before tail)"
        )

    prices_df = prices_df.tail(lookback_days)
    rets_df = np.log(prices_df / prices_df.shift(1)).dropna()

    if prices_df.shape[0] < (lookback_days * 0.9):
        if DEBUG and series_ranges:
            # Identify limiting start date and which assets cause it
            try:
                limiting_start = max(s for s, _, _ in series_ranges.values() if s is not None)
                limiting_names = [name for name, (s, _, _) in series_ranges.items() if s == limiting_start]
                print(f"Limiting start date: {limiting_start.date()} by {', '.join(limiting_names)}")
            except Exception:
                pass
        raise ValueError(
            f"After alignment only {prices_df.shape[0]} rows remain "
            f"(expected {lookback_days}). Data source may not have full history."
        )
    if rets_df.isna().any().any():
        raise ValueError("NaNs remained in returns after shift/drop; check data alignment.")
    if (prices_df <= 0).any().any():
        raise ValueError("Non-positive prices encountered; check source data.")

    values = {}
    for h in holdings:
        name = h['name']
        last_price = chf_close[name].iloc[-1]
        values[name] = h['position'] * last_price

    total_val = sum(values.values())
    print(f"Total portfolio value (CHF): {total_val:.2f}")

    # Weights (CHF)
    w = pd.Series({h["name"]: float(values[h["name"]]) / total_val for h in holdings})
    w = w.reindex(rets_df.columns).fillna(0.0)
    if not np.isclose(w.sum(), 1.0, atol=1e-6):
        raise ValueError(f"Weights must sum to 1. Got {w.sum():.6f}" "check postions input in holdings.")

    return rets_df, prices_df, w



def portfolio_risk(rets_df: pd.DataFrame, weights: pd.Series) -> dict:
    """
    Compute annualized vols, correlation, covariance, portfolio vol,
    marginal risk contribution (MRC), and percent risk contribution (PRC).
    """
    # Annualized stats
    cov_daily = rets_df.cov()
    cov_annual = cov_daily * 252.0
    vol_ann = rets_df.std() * np.sqrt(252.0)
    corr = rets_df.corr()

    # Align weights
    w = weights.reindex(rets_df.columns).astype(float)
    Sigma_w = cov_annual @ w
    port_var = float(w @ Sigma_w)
    port_vol = float(np.sqrt(port_var)) if port_var > 0 else 0.0

    # Contributions
    mrc = Sigma_w / port_vol if port_vol > 0 else Sigma_w * 0.0
    prc = (w * Sigma_w) / port_var if port_var > 0 else w * 0.0

    summary = pd.DataFrame({
        "Weight": w,
        "Vol_1Y_CHF": vol_ann,
        "MRC": mrc,           # ∂σ_p/∂w_i (absolute marginal contribution)
        "PRC_%": prc * 100.0  # percent contribution to total variance (sums ~100%)
    }).sort_values("PRC_%", ascending=False)

    return {
        "port_vol": port_vol,
        "cov_annual": cov_annual,
        "corr": corr,
        "vol_ann": vol_ann,
        "mrc": mrc,
        "prc": prc,
        "summary": summary,
    }

In [15]:
holdings0 = [
    {"name":"Unilever", "symbol":"ULVR.LON", "ccy":"GBP", "gbx":True,  "value_chf": 25000},
    {"name":"Shell",    "symbol":"SHEL.LON", "ccy":"GBP", "gbx":True,  "value_chf": 13000},
    {"name":"NatWest",  "symbol":"NWG.LON",  "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"Barclays", "symbol":"BARC.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"Tesco",    "symbol":"TSCO.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"SWDA",     "symbol":"SWDA.LON", "ccy":"GBP", "gbx":True,  "value_chf":  12000},
    {"name":"EMIM",     "symbol":"EMIM.LON", "ccy":"GBP", "gbx":True,  "value_chf":  8000},
    {"name":"IBM",      "symbol":"IBM",      "ccy":"USD", "gbx":False, "value_chf":  4000},
    {"name":"ERNS",     "symbol":"ERNS.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
]
holdings = [

    {"name":"EMIM",     "symbol":"EMIM.LSE", "ccy":"GBP", "gbx":True, "position": 100},
    {"name":"ERNS",     "symbol":"ERNS.LSE", "ccy":"GBP", "gbx":False, "position": 119},
    {"name":"IBM",     "symbol":"IBM.US", "ccy":"USD", "gbx":False, "position": 4},

    {"name":"SGLN",      "symbol":"SGLN.LSE",      "ccy":"GBP", "gbx":True, "position": 40},

    {"name":"VUAG",     "symbol":"VUAG.LSE", "ccy":"GBP", "gbx":False, "position": 28},
    {"name":"WSML",     "symbol":"WSML.LSE", "ccy":"USD", "gbx":False, "position": 313},
    {"name":"XMWX",     "symbol":"XMWX.LSE", "ccy":"GBP", "gbx":False, "position": 125,},


]

MAX_AGE = 0.01
PERIOD = 252
DEBUG = True

rets_df, prices_df, w = build_returns_matrix_in_chf(holdings, lookback_days=PERIOD, max_age=MAX_AGE, no_fx=False, DEBUG=DEBUG)

# print(f'rets_df: {rets_df.tail()}')
# print(f'prices_df: {prices_df.tail()}')

risk = portfolio_risk(rets_df, w)

# print(rets_df.tail())
print("Portfolio σ (annualized, CHF): {:.2%}".format(risk["port_vol"]))
print(risk["summary"].round({"Weight":3,"Vol_1Y_CHF":3,"MRC":3,"PRC_%":1, }))
# Optional:
print(f'CORRELATION:')
print(f'{risk["corr"].round(2)}')
print(f'COVARIANCE:')
print(f'{risk["cov_annual"]}')

looking at GBPCHF.FOREX ...
Cache file cache/GBPCHF.FOREX_2020-01-01.csv age: 0.37 hours (max age 0.01 hours)
Is fresh: False
Fetching GBPCHF.FOREX ...
 columns: ['Open', 'High', 'Low', 'Close', 'Adjusted_close', 'Volume']
writing new cache file
FX GBPCHF: 2020-01-01 → 2025-09-10 (1504 rows)
looking at USDCHF.FOREX ...
Cache file cache/USDCHF.FOREX_2020-01-01.csv age: 0.37 hours (max age 0.01 hours)
Is fresh: False
Fetching USDCHF.FOREX ...
 columns: ['Open', 'High', 'Low', 'Close', 'Adjusted_close', 'Volume']
writing new cache file
FX GBPCHF: 2020-01-01 → 2025-09-10 (1504 rows)
looking at USDCHF.FOREX ...
Cache file cache/USDCHF.FOREX_2020-01-01.csv age: 0.37 hours (max age 0.01 hours)
Is fresh: False
Fetching USDCHF.FOREX ...


/var/folders/mp/2szq6ct92zl4hbdd0y18rqyw0000gn/T/ipykernel_71611/1366045741.py:85: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  if not a.fillna(method="ffill").equals(b.fillna(method="ffill")):


 columns: ['Open', 'High', 'Low', 'Close', 'Adjusted_close', 'Volume']
writing new cache file
shifting fx series()
Applied USDCHF T+1 shift
FX USDCHF: 2020-01-01 → 2025-09-10 (1500 rows)
FX keys loaded: ['GBPCHF', 'USDCHF']
Per-asset CHF ranges:
looking at EMIM.LSE ...
Cache file cache/EMIM.LSE_2020-01-01.csv age: 0.37 hours (max age 0.01 hours)
Is fresh: False
Fetching EMIM.LSE ...


/var/folders/mp/2szq6ct92zl4hbdd0y18rqyw0000gn/T/ipykernel_71611/1366045741.py:85: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  if not a.fillna(method="ffill").equals(b.fillna(method="ffill")):


 columns: ['Open', 'High', 'Low', 'Close', 'Adjusted_close', 'Volume']
writing new cache file
  EMIM: 2020-01-02 → 2025-09-10 (1437 rows)
looking at ERNS.LSE ...
Cache file cache/ERNS.LSE_2020-01-01.csv age: 0.37 hours (max age 0.01 hours)
Is fresh: False
Fetching ERNS.LSE ...


...

/var/folders/mp/2szq6ct92zl4hbdd0y18rqyw0000gn/T/ipykernel_71611/1366045741.py:85: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  if not a.fillna(method="ffill").equals(b.fillna(method="ffill")):


 columns: ['Open', 'High', 'Low', 'Close', 'Adjusted_close', 'Volume']
writing new cache file
    WSML: FX ffill filled 1 missing FX points on price dates
  WSML: 2020-01-02 → 2025-09-10 (1437 rows)
looking at XMWX.LSE ...
Cache file cache/XMWX.LSE_2020-01-01.csv age: 0.37 hours (max age 0.01 hours)
Is fresh: False
Fetching XMWX.LSE ...


/var/folders/mp/2szq6ct92zl4hbdd0y18rqyw0000gn/T/ipykernel_71611/1366045741.py:85: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  if not a.fillna(method="ffill").equals(b.fillna(method="ffill")):


 columns: ['Open', 'High', 'Low', 'Close', 'Adjusted_close', 'Volume']
writing new cache file
  XMWX: 2024-08-27 → 2025-09-10 (264 rows)
Common window: 2024-08-27 → 2025-09-09 (255 rows before tail)
Total portfolio value (CHF): 28333.73
Portfolio σ (annualized, CHF): 9.60%
      Weight  Vol_1Y_CHF    MRC  PRC_%
ERNS   0.461       0.078  0.061   29.1
XMWX   0.136       0.152  0.138   19.5
EMIM   0.119       0.171  0.144   17.8
VUAG   0.099       0.187  0.149   15.4
WSML   0.077       0.209  0.130   10.3
SGLN   0.080       0.169  0.057    4.8
IBM    0.029       0.293  0.102    3.1
CORRELATION:
      EMIM  ERNS   IBM  SGLN  VUAG  WSML  XMWX
EMIM  1.00  0.51  0.23  0.16  0.70  0.56  0.79
ERNS  0.51  1.00  0.18  0.33  0.47  0.13  0.58
IBM   0.23  0.18  1.00  0.14  0.15  0.26  0.22
SGLN  0.16  0.33  0.14  1.00  0.01 -0.02  0.16
VUAG  0.70  0.47  0.15  0.01  1.00  0.63  0.77
WSML  0.56  0.13  0.26 -0.02  0.63  1.00  0.68
XMWX  0.79  0.58  0.22  0.16  0.77  0.68  1.00
COVARIANCE:
          EMI

/var/folders/mp/2szq6ct92zl4hbdd0y18rqyw0000gn/T/ipykernel_71611/1366045741.py:85: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  if not a.fillna(method="ffill").equals(b.fillna(method="ffill")):
